In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

In [2]:
import os

print(os.getcwd())

C:\Users\SATYA\OneDrive\Desktop\Week2-Work-Data-Validation\notebooks


In [4]:
import os

print(os.listdir())

['.ipynb_checkpoints', 'Data_Validation.ipynb']


In [3]:
import pandas as pd

# Load the Week 1 output (wide format: date x category)
wide = pd.read_csv(
    "../data/monthly_demand_clean.csv",
    index_col=0,
    parse_dates=True
)

# Rename the index
wide.index.name = "Date"

# Display information
print("Shape:", wide.shape)
print("Categories found:", list(wide.columns))

# Show first 5 rows
wide.head()

Shape: (54, 5)
Categories found: ['CPU', 'Mother Board', 'RAM', 'Storage', 'Video Card']


,CPU,Mother Board,RAM,Storage,Video Card
Date,,,,,
2013-06-01,NaN,NaN,NaN,119.0,NaN
2013-07-01,NaN,NaN,NaN,0.0,NaN
2013-08-01,NaN,NaN,NaN,0.0,NaN
2013-09-01,NaN,NaN,NaN,0.0,NaN
2013-10-01,NaN,NaN,NaN,0.0,NaN


In [4]:
# Create the "long" version yourself since Week 1 didn't export it
long_df = wide.reset_index().melt(
    id_vars="Date", var_name="CategoryName", value_name="Demand"
)
long_df = long_df.sort_values(["CategoryName", "Date"]).reset_index(drop=True)

# Save it so your teammates can use it too
long_df.to_csv("monthly_demand_clean_long.csv", index=False)

print("Long shape:", long_df.shape)
long_df.head()

Long shape: (270, 3)


,Date,CategoryName,Demand
0,2013-06-01,CPU,NaN
1,2013-07-01,CPU,NaN
2,2013-08-01,CPU,NaN
3,2013-09-01,CPU,NaN
4,2013-10-01,CPU,NaN


In [5]:
# Verify category counts — every category should have the SAME number of months
# (because Week 1 built a complete monthly timeline for every category)
counts = long_df.groupby("CategoryName")["Date"].count()
print(counts)

expected_months = wide.shape[0]
assert (counts == expected_months).all(), "Some categories are missing months!"
print(f"✅ All {len(counts)} categories have {expected_months} months each.")

CategoryName
CPU             54
Mother Board    54
RAM             54
Storage         54
Video Card      54
Name: Date, dtype: int64
✅ All 5 categories have 54 months each.


In [6]:
# 1. Duplicate months per category (there should be ZERO)
dupes = long_df[long_df.duplicated(subset=["CategoryName", "Date"], keep=False)]
print("Duplicate rows found:", len(dupes))
dupes

Duplicate rows found: 0


,Date,CategoryName,Demand


In [7]:
# 2. Negative demand values (impossible in real life — flag if found)
negatives = long_df[long_df["Demand"] < 0]
print("Negative demand rows found:", len(negatives))
negatives

Negative demand rows found: 0


,Date,CategoryName,Demand


In [8]:
# 3. Remaining missing values (leading/trailing gaps Week 1 intentionally left)
missing = long_df[long_df["Demand"].isna()]
print("Missing rows found:", len(missing))
print(missing.groupby("CategoryName").size())
missing

Missing rows found: 96
CategoryName
CPU             23
Mother Board    23
RAM             27
Video Card      23
dtype: int64


,Date,CategoryName,Demand
0,2013-06-01,CPU,NaN
1,2013-07-01,CPU,NaN
2,2013-08-01,CPU,NaN
3,2013-09-01,CPU,NaN
4,2013-10-01,CPU,NaN
...,...,...,...
234,2014-12-01,Video Card,NaN
235,2015-01-01,Video Card,NaN
236,2015-02-01,Video Card,NaN
237,2015-03-01,Video Card,NaN


In [9]:
# Extra check: are all remaining missing values at the START or END of each series (expected),
# or somewhere in the MIDDLE (a bug)?
def check_gap_position(group):
    group = group.sort_values("Date").reset_index(drop=True)
    na_idx = group[group["Demand"].isna()].index
    if len(na_idx) == 0:
        return "no gaps"
    n = len(group)
    if all(i < n * 0.15 or i > n * 0.85 for i in na_idx):
        return "edges only (expected)"
    return "⚠️ MIDDLE gap — investigate"

gap_report = long_df.groupby("CategoryName").apply(check_gap_position)
print(gap_report)

CategoryName
CPU             ⚠️ MIDDLE gap — investigate
Mother Board    ⚠️ MIDDLE gap — investigate
RAM             ⚠️ MIDDLE gap — investigate
Storage                             no gaps
Video Card      ⚠️ MIDDLE gap — investigate
dtype: str
